# Embeddings con tu proyecto

**Sesión 6 · IA Intermedio · EstudIA 2026**

Hoy pruebas los embeddings con tus propios ejemplos, según tu proyecto:

| Camino | Para quién | Qué haces |
|---|---|---|
| **A1. Texto** | Tu proyecto clasifica textos | Zero-shot con embeddings, embeddings + regresión logística con validación cruzada, y una búsqueda de ejemplos casi iguales con etiquetas distintas |
| **A2. Imagen** | Tu proyecto clasifica imágenes | Lo mismo con embeddings de imágenes, y las fotos más parecidas a cada foto |
| **B. Práctica** | Todavía no tienes ejemplos | Haces A1 con los 30 mensajes ciudadanos de las sesiones 4 y 5 |

Al final de A1 y de B hay una tabla para comparar con lo que mediste en las sesiones anteriores. Cuando termines,
**entrega el notebook** desde la última celda. No importa si el modelo acierta.

**Antes de empezar.** *Archivo › Guardar una copia en Drive*. No necesitas GPU.

In [ ]:
# Preparación: librerías. Ejecútala una vez.
import os, re, time, zipfile, urllib.request
from collections import Counter
ARCHIVOS = os.environ.get("ESTUDIA_ARCHIVOS", "https://storage.googleapis.com/estud-ia-course-estudia-lab/s06")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

try:
    from sentence_transformers import SentenceTransformer
except ImportError:  # Colab ya la trae; fuera de Colab puede faltar
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sentence-transformers"], check=True)
    from sentence_transformers import SentenceTransformer

GPU = torch.cuda.is_available()
print("Listo.", f"GPU: {torch.cuda.get_device_name(0)}." if GPU else "Sin GPU: alcanza para todo el notebook.")

In [ ]:
# @title Funciones auxiliares: números, tokens, evaluar, validación cruzada y zero-shot
def descargar(nombre, url=None):
    """Baja un archivo (de la carpeta pública del curso, o de `url`) solo la primera vez y devuelve su ruta."""
    ruta = os.path.join("archivos_curso", os.path.basename(nombre))
    if not os.path.exists(ruta):
        os.makedirs(os.path.dirname(ruta), exist_ok=True)
        urllib.request.urlretrieve(url or f"{ARCHIVOS}/{nombre}", ruta)
    return ruta

def miles(n):
    """12345 -> "12.345", como escribimos los números en español."""
    return f"{n:,}".replace(",", ".")

def pct(x, decimales=1):
    """0.958 -> "95,8 %"."""
    return f"{100 * x:.{decimales}f} %".replace(".", ",")

def tokens(texto):
    """Parte un texto en palabras y signos, en minúsculas: "No me gustó." -> ["no", "me", "gustó", "."]."""
    return re.findall(r"\w+|[^\w\s]", str(texto).lower())

def evaluar(verdad, prediccion, titulo="Tu modelo"):
    """Compara las predicciones con las etiquetas reales y con la línea base de la clase más frecuente."""
    verdad, prediccion = [str(v) for v in verdad], [str(p) for p in prediccion]
    clases = sorted(set(verdad))
    frecuente = max(clases, key=verdad.count)
    def balanceada(p):
        return float(np.mean([np.mean([pi == c for vi, pi in zip(verdad, p) if vi == c]) for c in clases]))
    print(f"{titulo} · {len(verdad)} ejemplos, {len(clases)} clases")
    print(f"  Exactitud:            {pct(np.mean([v == p for v, p in zip(verdad, prediccion)]))}   "
          f"(línea base “siempre {frecuente}”: {pct(np.mean([v == frecuente for v in verdad]))})")
    print(f"  Exactitud balanceada: {pct(balanceada(prediccion))}   (línea base: {pct(balanceada([frecuente] * len(verdad)))})")
    todas = clases + sorted(set(prediccion) - set(clases))
    fig, eje = plt.subplots(figsize=(1.1 * len(todas) + 2.5, 0.9 * len(clases) + 1.8))
    ConfusionMatrixDisplay(confusion_matrix(verdad, prediccion, labels=todas), display_labels=todas).plot(
        ax=eje, cmap="Purples", colorbar=False, xticks_rotation=45)
    eje.set_xlabel("Lo que respondió"); eje.set_ylabel("La etiqueta real"); eje.set_title(titulo)
    eje.set_ylim(len(clases) - 0.5, -0.5)
    plt.tight_layout(); plt.show()

def _pliegues(etiquetas):
    n = int(min(5, np.bincount(np.unique(etiquetas, return_inverse=True)[1]).min()))
    if n < 2:
        print("Necesitas al menos 2 ejemplos de cada etiqueta.")
    return n


def bolsa_con_validacion_cruzada(textos, etiquetas):
    """Bolsa de palabras + regresión logística, medida con validación cruzada (la de la sesión 5)."""
    etiquetas = np.array([str(e) for e in etiquetas])
    n = _pliegues(etiquetas)
    if n < 2:
        return None
    modelo = make_pipeline(TfidfVectorizer(tokenizer=tokens, lowercase=False, token_pattern=None, ngram_range=(1, 2)),
                           LogisticRegression(max_iter=3000, C=10, class_weight="balanced"))
    return cross_val_predict(modelo, list(textos), etiquetas, cv=StratifiedKFold(n, shuffle=True, random_state=0))

def embeddings_con_validacion_cruzada(X, etiquetas, C=10):
    """Regresión logística sobre embeddings ya calculados (una fila por ejemplo), con validación cruzada."""
    etiquetas = np.array([str(e) for e in etiquetas])
    n = _pliegues(etiquetas)
    if n < 2:
        return None
    return cross_val_predict(LogisticRegression(max_iter=5000, C=C, class_weight="balanced"), X, etiquetas,
                             cv=StratifiedKFold(n, shuffle=True, random_state=0))

def zero_shot(X, etiquetas_posibles, D):
    """A cada fila de X le da la etiqueta cuya descripción (fila de D) queda más cerca."""
    return [etiquetas_posibles[i] for i in np.argmax(X @ D.T, axis=1)]

def parecidos_con_otra_etiqueta(X, ejemplos, etiquetas, cuantos=5):
    """Los pares de ejemplos más parecidos entre sí que tienen etiquetas distintas: posibles etiquetas dudosas."""
    S = X @ X.T
    pares = [(S[i, j], i, j) for i in range(len(X)) for j in range(i + 1, len(X)) if etiquetas[i] != etiquetas[j]]
    for s, i, j in sorted(pares, reverse=True)[:cuantos]:
        print(f"{s:.2f}".replace(".", ","), f"· [{etiquetas[i]}] {str(ejemplos[i])[:70]}")
        print("     ", f"[{etiquetas[j]}] {str(ejemplos[j])[:70]}")

In [ ]:
MODELO_EMBEDDINGS = "intfloat/multilingual-e5-small"   # o "intfloat/multilingual-e5-base", "BAAI/bge-m3"…
PREFIJO = "query: " if "e5" in MODELO_EMBEDDINGS else ""
_e = SentenceTransformer(MODELO_EMBEDDINGS, device="cuda" if GPU else "cpu")

def embeber(textos):
    """Un vector por texto, de largo 1, con el prefijo que pide el modelo. Pasa los textos a minúsculas."""
    return _e.encode([PREFIJO + str(t).lower() for t in textos], batch_size=32, normalize_embeddings=True,
                     show_progress_bar=False)

---
## A1. Texto

### Tus ejemplos

Los mismos de las sesiones 4 y 5: escríbelos en `EJEMPLOS` o sube `ejemplos.csv` (columnas `entrada` y `etiqueta`).
Sin datos personales. En `DESCRIPCIONES`, escribe una frase corta que describa cada etiqueta: la usa el zero-shot.

In [ ]:
EJEMPLOS = [
    ("Escribe aquí un ejemplo real de tu proyecto.", "etiqueta_1"),
    ("Y otro, con otra etiqueta.", "etiqueta_2"),
]
DESCRIPCIONES = {
    "etiqueta_1": "una frase que describe la etiqueta 1",
    "etiqueta_2": "una frase que describe la etiqueta 2",
}

if os.path.exists("ejemplos.csv"):
    tabla = pd.read_csv("ejemplos.csv")
    EJEMPLOS = list(zip(tabla["entrada"].astype(str), tabla["etiqueta"].astype(str)))
mis_textos = not EJEMPLOS[0][0].startswith("Escribe aquí")
if mis_textos:
    textos, etiquetas = [e for e, _ in EJEMPLOS], [str(et) for _, et in EJEMPLOS]
    ETIQUETAS = sorted(set(etiquetas))
    print(f"{len(textos)} ejemplos. Etiquetas:", pd.Series(etiquetas).value_counts().to_dict())
    faltan = [e for e in ETIQUETAS if e not in DESCRIPCIONES]
    if faltan:
        print("Sin descripción (uso el nombre de la etiqueta):", faltan)
    X_mios = embeber(textos)
else:
    print("Camino A1: escribe tus ejemplos en EJEMPLOS (o sube ejemplos.csv) y vuelve a ejecutar.")

### 1. Zero-shot con embeddings

Sin entrenar nada: a cada ejemplo le damos la etiqueta cuya descripción queda más cerca. Prueba a cambiar las
descripciones y vuelve a ejecutar.

In [ ]:
if mis_textos:
    D_mias = embeber([DESCRIPCIONES.get(e, e.replace("_", " ")) for e in ETIQUETAS])
    evaluar(etiquetas, zero_shot(X_mios, ETIQUETAS, D_mias), "Zero-shot con embeddings")

### 2. Embeddings + regresión logística, con validación cruzada

La misma validación cruzada de la sesión 5: cada ejemplo se predice con un modelo que no lo vio. Al lado, la bolsa de
palabras con los mismos ejemplos.

In [ ]:
if mis_textos:
    pred = embeddings_con_validacion_cruzada(X_mios, etiquetas)
    if pred is not None:
        evaluar(etiquetas, pred, "Embeddings + regresión logística")
        evaluar(etiquetas, bolsa_con_validacion_cruzada(textos, etiquetas), "Bolsa de palabras")

### 3. Ejemplos casi iguales con etiquetas distintas

Los pares de ejemplos más parecidos que tienen etiquetas distintas. A veces son casos difíciles de verdad; a veces
son etiquetas que hay que revisar, o una definición de etiqueta que hay que precisar.

In [ ]:
if mis_textos:
    parecidos_con_otra_etiqueta(X_mios, textos, etiquetas)

### Tu tabla de comparación

Completa con lo que mediste (doble clic aquí). Usa los mismos ejemplos en todas las filas.

| Modelo | Exactitud | Exactitud balanceada |
|---|---|---|
| Línea base (la clase más frecuente) | | |
| Tu modelo base (sesión 4) | | |
| Bolsa de palabras (sesión 5) | | |
| Promedio de vectores de fastText (sesión 5) | | |
| Zero-shot con embeddings (hoy) | | |
| Embeddings + regresión logística (hoy) | | |

**Lo que observé** (responde en dos o tres líneas):

- ¿Cuál fila gana? ¿Por cuánto? Con tus pocos ejemplos, ¿la diferencia es clara o podría ser suerte?
- ¿Qué encontraste en los pares casi iguales con etiquetas distintas? ¿Cambiarías alguna etiqueta o alguna definición?

---
## A2. Imagen

Sube un archivo `mis_fotos.zip` con una carpeta por etiqueta (por ejemplo, `mis_fotos/sana/…jpg` y
`mis_fotos/enferma/…jpg`), sin personas reconocibles. Usamos
[`facebook/dinov2-small`](https://huggingface.co/facebook/dinov2-small) (licencia Apache 2.0), un modelo entrenado
para dar buenos embeddings de imágenes. También puedes escribir en `MODELO_IMAGEN` el cuerpo de tu modelo base.

In [ ]:
MODELO_IMAGEN = "facebook/dinov2-small"

mis_fotos = os.path.exists("mis_fotos.zip")
if mis_fotos:
    from PIL import Image
    from transformers import AutoImageProcessor, AutoModel
    with zipfile.ZipFile("mis_fotos.zip") as z:
        z.extractall(".")
    rutas = sorted(os.path.join(r, f) for r, _, fs in os.walk("mis_fotos") for f in fs
                   if f.lower().endswith((".jpg", ".jpeg", ".png", ".webp")))
    etiquetas_img = [os.path.basename(os.path.dirname(r)) for r in rutas]
    fotos = [Image.open(r).convert("RGB") for r in rutas]
    procesador = AutoImageProcessor.from_pretrained(MODELO_IMAGEN)
    modelo_img = AutoModel.from_pretrained(MODELO_IMAGEN).eval()
    with torch.no_grad():
        vectores = []
        for i in range(0, len(fotos), 16):
            salida = modelo_img(**procesador(images=fotos[i:i + 16], return_tensors="pt"))
            v = salida.pooler_output if getattr(salida, "pooler_output", None) is not None else salida.last_hidden_state[:, 0]
            vectores.append(torch.nn.functional.normalize(v.flatten(1), dim=-1))
    X_fotos = torch.cat(vectores).numpy()
    print(f"{len(fotos)} fotos, {len(set(etiquetas_img))} etiquetas: {dict(Counter(etiquetas_img))}. "
          f"Cada foto es un vector de {X_fotos.shape[1]} números.")
    evaluar(etiquetas_img, embeddings_con_validacion_cruzada(X_fotos, etiquetas_img), "Embeddings de imagen + regresión logística")
else:
    print("Camino A2: sube mis_fotos.zip y vuelve a ejecutar.")

### Las fotos más parecidas

La primera foto de cada fila es una de las tuyas; las otras cuatro son las más cercanas en el espacio de los
embeddings, con su etiqueta.

In [ ]:
if mis_fotos:
    rng = np.random.default_rng(0)
    elegidas = rng.choice(len(fotos), size=min(4, len(fotos)), replace=False)
    fig, ejes = plt.subplots(len(elegidas), 5, figsize=(11, 2.4 * len(elegidas)))
    for fila, i in zip(np.atleast_2d(ejes), elegidas):
        cercanas = [j for j in np.argsort(-(X_fotos @ X_fotos[i])) if j != i][:4]
        for eje, j in zip(fila, [i] + cercanas):
            eje.imshow(fotos[j].resize((160, 160))); eje.axis("off")
            eje.set_title(("Esta: " if j == i else "") + etiquetas_img[j], fontsize=9,
                          color="black" if etiquetas_img[j] == etiquetas_img[i] else "#CB318E")
    plt.tight_layout(); plt.show()

**Lo que observé** (doble clic aquí y responde): ¿cómo queda la regresión logística sobre embeddings frente a tu modelo
ajustado de la sesión 5? ¿Las fotos más parecidas comparten etiqueta? Cuando no, ¿qué tienen en común?

---
## B. Práctica: los mensajes ciudadanos

Los 30 mensajes de las sesiones 4 y 5, cinco por tema. Las descripciones son las mismas frases que usó el zero-shot de
la sesión 4. Para el zero-shot te falta una línea: convertir esas frases en vectores (si te atascas, abre la **pista**
que está debajo de la celda).

In [ ]:
MENSAJES = [
    ("El semáforo de la 80 con la 33 lleva tres días apagado y se forman unos trancones terribles.", "movilidad"),
    ("El metro iba tan lleno esta mañana que no pude subirme en tres trenes.", "movilidad"),
    ("Los buses de la ruta del barrio dejaron de pasar después de las ocho de la noche.", "movilidad"),
    ("Hay un hueco enorme en la vía principal y ya se han caído dos motociclistas.", "movilidad"),
    ("No hay señalización en la nueva ciclorruta y los carros se parquean encima.", "movilidad"),
    ("Anoche robaron dos celulares en el paradero del parque y no había ningún policía.", "seguridad"),
    ("Pusieron cámaras en la ciclorruta, pero siguen robando bicicletas.", "seguridad"),
    ("En la esquina de la cancha venden droga a plena luz del día.", "seguridad"),
    ("Las luces del puente peatonal están dañadas y da miedo pasar de noche.", "seguridad"),
    ("Unos muchachos están cobrando vacuna a los tenderos de la cuadra.", "seguridad"),
    ("En el centro de salud del barrio no hay citas para medicina general hasta el próximo mes.", "salud"),
    ("Llevo tres horas esperando en urgencias con mi mamá y nadie nos atiende.", "salud"),
    ("La farmacia de la EPS no tiene la insulina desde hace dos semanas.", "salud"),
    ("Hay muchos casos de dengue en el sector y nadie ha venido a fumigar.", "salud"),
    ("Necesitamos una jornada de vacunación para los niños de la vereda.", "salud"),
    ("La escuela de la vereda no tiene profesor de matemáticas desde marzo.", "educación"),
    ("Al colegio no ha llegado el refrigerio de los estudiantes en todo el mes.", "educación"),
    ("Los computadores de la sala de informática del colegio llevan un año dañados.", "educación"),
    ("No hay cupos en el preescolar del barrio y tengo que trabajar.", "educación"),
    ("El techo del salón se cae a pedazos cuando llueve y los niños reciben clase así.", "educación"),
    ("Hace una semana se fue el agua en toda la cuadra y nadie da razón.", "servicios públicos"),
    ("La factura de la luz llegó el doble de lo normal sin ninguna explicación.", "servicios públicos"),
    ("Hay una fuga de gas en el contador del edificio y huele muy fuerte.", "servicios públicos"),
    ("El alcantarillado se rebosa cada vez que llueve y el agua entra a las casas.", "servicios públicos"),
    ("Se cayó un cable de energía en el andén y está echando chispas.", "servicios públicos"),
    ("Están quemando basura al lado de la quebrada y el humo llega a todas las casas.", "medio ambiente"),
    ("Talaron los árboles del parque sin ningún permiso.", "medio ambiente"),
    ("Una fábrica está botando aguas negras al río.", "medio ambiente"),
    ("La calidad del aire está pésima y los niños tienen tos todo el día.", "medio ambiente"),
    ("Dejan escombros y colchones viejos en la orilla de la quebrada.", "medio ambiente"),
]
DESCRIPCIONES_B = {"movilidad": "tránsito y transporte", "seguridad": "seguridad y delincuencia",
                   "salud": "salud y hospitales", "educación": "educación y colegios",
                   "servicios públicos": "agua, luz o gas", "medio ambiente": "medio ambiente y contaminación"}
textos_b, temas_b = [m for m, _ in MENSAJES], [t for _, t in MENSAJES]
TEMAS = sorted(DESCRIPCIONES_B)
X_b = embeber(textos_b)
FRASES_B = [f"Este mensaje trata sobre {DESCRIPCIONES_B[t]}." for t in TEMAS]

# Completa: convierte FRASES_B en vectores, con la misma función que usamos para los mensajes.
D_b = ...

if D_b is ...:
    print("Completa la línea de D_b y vuelve a ejecutar. Si te atascas, abre la pista de abajo.")
else:
    evaluar(temas_b, zero_shot(X_b, TEMAS, D_b), "Zero-shot con embeddings, 30 mensajes")
evaluar(temas_b, embeddings_con_validacion_cruzada(X_b, temas_b), "Embeddings + regresión logística, 30 mensajes")
parecidos_con_otra_etiqueta(X_b, textos_b, temas_b, cuantos=3)

<details><summary>Pista</summary>

El zero-shot compara el vector de cada mensaje con el vector de cada descripción y elige la más cercana. Para que esa
comparación tenga sentido, las descripciones tienen que pasar por el mismo modelo que los mensajes. Mira cómo se
calculó `X_b`, los vectores de los mensajes, dos líneas más arriba, y haz lo mismo con `FRASES_B`.
</details>

### Tu tabla de comparación

Completa con lo que mediste hoy (doble clic aquí). Las tres primeras filas ya las mediste en las sesiones 4 y 5.

| Modelo | Exactitud con los 30 mensajes |
|---|---|
| Zero-shot que lee texto y etiqueta (sesión 4) | 63 % |
| Bolsa de palabras (sesión 5) | 13 % |
| Promedio de vectores de fastText (sesión 5) | 60 % |
| Zero-shot con embeddings (hoy) | |
| Embeddings + regresión logística (hoy) | |

**Lo que observé** (responde en dos o tres líneas): ¿cuál fila gana? Con cinco ejemplos por tema, ¿qué aportan los
embeddings que no tenía la bolsa de palabras?

---
## Entrega

Ejecuta las dos celdas siguientes. La primera solo prepara la función; la segunda lee este notebook con sus
resultados y lo envía a la plataforma. Te pide tu clave la primera vez.

Si algo falla, descarga el notebook (*Archivo › Descargar › Descargar .ipynb*) y súbelo en la plataforma, en la
actividad “Entrega: embeddings con tu proyecto”.

In [ ]:
# @title Función para entregar el notebook (no hace falta cambiarla)
import getpass, json, urllib.error

PLATAFORMA = os.environ.get("ESTUDIA_PLATAFORMA", "https://estud-ia.eslopezag.com")
_CLAVE = os.environ.get("ESTUDIA_CLAVE", "")
ACTIVIDAD = "“Entrega: embeddings con tu proyecto”"

def _este_notebook():
    """El notebook tal como se ve en Colab, con sus resultados. Fuera de Colab, el archivo de ESTUDIA_NOTEBOOK."""
    if os.environ.get("ESTUDIA_NOTEBOOK"):
        with open(os.environ["ESTUDIA_NOTEBOOK"], encoding="utf-8") as f:
            return json.load(f)
    try:
        from google.colab import _message
        return _message.blocking_request("get_ipynb", timeout_sec=60)["ipynb"]
    except Exception:
        return None

def entregar():
    """Envía este notebook, con lo que ejecutaste, a la plataforma del curso."""
    global _CLAVE
    cuaderno = _este_notebook()
    if not cuaderno:
        print("No pude leer el notebook desde aquí. Descárgalo (Archivo › Descargar › Descargar .ipynb) y súbelo "
              f"en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    if not _CLAVE:
        _CLAVE = getpass.getpass(f"Pega tu clave (plataforma › {ACTIVIDAD} › Copiar) y oprime Enter: ").strip()
    req = urllib.request.Request(f"{PLATAFORMA}/api/entregas/s06", data=json.dumps({"notebook": cuaderno}).encode(),
                                 method="POST", headers={"Content-Type": "application/json",
                                                         "Authorization": f"Bearer {_CLAVE}",
                                                         "User-Agent": "EstudIA-notebook/1.0"})
    try:
        with urllib.request.urlopen(req, timeout=120) as r:
            res = json.loads(r.read())
    except urllib.error.HTTPError as e:
        res = json.loads(e.read() or b"{}")
        if e.code == 401:
            _CLAVE = ""
        print("No se pudo entregar:", res.get("error", f"error {e.code}"))
        print(f"Si no se arregla, descarga el notebook y súbelo en la plataforma, en la actividad {ACTIVIDAD}.")
        return
    except urllib.error.URLError:
        print("No hay conexión con la plataforma. Revisa tu internet o pregúntale al profesor si está encendida.")
        return
    modelos = ", ".join(res["modelos"]) or "ninguno"
    print(f"Entrega recibida. Celdas de código ejecutadas: {res['ejecutadas']} de {res['celdas']}. "
          f"Modelos de Hugging Face que encontré: {modelos}.")
    for aviso in res.get("avisos", []):
        print("Ojo:", aviso)

In [ ]:
entregar()